# WikiArt Dataset Analysis

**Author:** ML / Computer-Vision engineering assessment submission
**Scope:** structural analysis and data-quality audit of a 27-style WikiArt
painting corpus (~81k images) with two accompanying label files,
`classes.csv` and `wclasses.csv`.

---

## Source-of-truth hierarchy (read this first)

This analysis follows one explicit, documented hierarchy, established after
directly inspecting the data (not inferred from folder names alone):

1. **Filesystem (the 27 style folders) — authoritative image inventory.**
   It is the only artifact that exactly represents the images that exist on
   disk, and the style is unambiguous from the folder. Verified: the folder
   name equals the primary label in `classes.csv` for **every** overlapping
   row (zero mismatches).
2. **`classes.csv` — primary human-readable metadata.** Artist name, pixel
   dimensions, perceptual hash (`phash`), the official `train`/`test` split,
   and style label. Joined onto the inventory; it is an *enrichment*, never
   the inventory itself (it covers a strict subset of the images).
3. **`wclasses.csv` — validation / reconciliation only.** Numeric-ID encoding
   with **no legend file present in the dataset**, so its artist/genre IDs
   cannot be resolved to names. Used to cross-check coverage; inconsistencies
   are reported transparently rather than silently discarded or overwritten.

### Key facts established by direct inspection
- The `genre` column in `classes.csv` actually holds **style** labels (it is
  mislabelled at source), stored as a Python-list literal, e.g.
  `['Impressionism']`.
- `wclasses.csv`'s `genre` is a *different, genuine* concept (11 categories),
  distinct from its 27-way `style`. Because there is no legend, those IDs stay
  opaque and are reported as-is.
- Non-ASCII artist names are mangled by **two different mojibake encodings**
  (one on disk, another in `classes.csv`). Exact string joins silently drop
  every accented filename, so an encoding-robust join key is required — see
  `normalize_key`.

### How to run
Execute top-to-bottom. Requires the packages in `requirements.txt`. The full
image-integrity scan decodes every image and is the slow step; set
`INTEGRITY_SAMPLE` below to an integer for a fast partial pass.

> **Structure:** Sections 1–6 are the **required** analyses. Section 7 contains
> **optional enhancements** (perceptual-hash near-duplicate detection),
> presented only after every required analysis is complete.

## 0. Imports and configuration

In [ ]:
from __future__ import annotations

import ast
import os
import re
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

# ===========================================================================
# DATASET LOCATION  ---  the only path you need to configure
# ===========================================================================
# The WikiArt dataset lives OUTSIDE this repository. Point DATASET_ROOT at the
# folder that directly contains ``classes.csv``, ``wclasses.csv`` and the 27
# style sub-folders. Configure it in either of two ways:
#   1. Set an environment variable before launching Jupyter:
#          export WIKIART_DATASET_ROOT=/path/to/wikiart      (macOS/Linux)
#          set    WIKIART_DATASET_ROOT=C:\path\to\wikiart    (Windows)
#   2. Or simply edit the fallback path on the next line.
DATASET_ROOT = Path(
    os.environ.get("WIKIART_DATASET_ROOT", "data/wikiart")
).expanduser().resolve()

# --- Reproducibility -------------------------------------------------------
RANDOM_SEED: int = 42
RNG = np.random.default_rng(RANDOM_SEED)

# --- Configuration ---------------------------------------------------------
IMAGE_EXTENSIONS: frozenset[str] = frozenset({".jpg", ".jpeg", ".png"})

# Set to an int (e.g. 4000) for a fast partial integrity scan; None = full.
INTEGRITY_SAMPLE: int | None = None

# Detect truncated images instead of silently loading partial pixel data.
ImageFile.LOAD_TRUNCATED_IMAGES = False

# --- Publication-quality plotting defaults ---------------------------------
mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
pd.set_option("display.max_colwidth", 80)

## 1. Locate the dataset and build the authoritative inventory

The dataset lives outside the repository and is located through the single
`DATASET_ROOT` variable configured in Section 0 (via the `WIKIART_DATASET_ROOT`
environment variable or by editing that one line). `validate_dataset_root`
fails fast with an actionable message if it is not set correctly.

`build_inventory` walks the 27 style folders with `pathlib` and produces **one
row per image that physically exists on disk** — the source of truth for the
image set and its style label.

In [ ]:
def validate_dataset_root(root: Path) -> Path:
    """Confirm ``root`` points at the dataset; raise a helpful error if not."""
    if not (root / "classes.csv").exists():
        raise FileNotFoundError(
            f"classes.csv was not found under DATASET_ROOT={root!s}.\n"
            "Set the WIKIART_DATASET_ROOT environment variable, or edit "
            "DATASET_ROOT in the configuration cell, to the folder that "
            "directly contains classes.csv, wclasses.csv and the 27 style "
            "sub-folders."
        )
    return root


def normalize_key(relative_path: str) -> str:
    """Return an encoding-robust join key for a dataset-relative image path.

    Non-ASCII artist names are mangled by *two different* mojibake encodings
    (one on disk, another in ``classes.csv``), so exact string joins silently
    drop every accented filename. Dropping all non-ASCII bytes and keeping only
    lowercase alphanumerics collapses both corruptions to the same key, e.g.
    both encodings of ``eug<...>ne-grasset`` reduce to ``eugnegrasset``.

    Unicode NFKD folding is deliberately avoided: it transliterates the two
    corrupt byte sequences inconsistently (introducing ``o``/``14`` artifacts)
    and therefore fails to reconcile them. Any collisions this key introduces
    are measured and reported (Section 2), not assumed to be zero.
    """
    ascii_only = relative_path.encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]", "", ascii_only.lower())


def build_inventory(project_root: Path) -> pd.DataFrame:
    """Scan the 27 style folders; return the authoritative image inventory.

    One row per on-disk image. ``style`` comes from the folder name (the
    filesystem is authoritative for which images exist and their style).
    """
    records: list[dict[str, object]] = []
    style_dirs = sorted(p for p in project_root.iterdir() if p.is_dir())
    for style_dir in tqdm(style_dirs, desc="Scanning style folders"):
        style = style_dir.name.replace("_", " ")
        for path in style_dir.iterdir():
            if path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            records.append({
                "relative_path": f"{style_dir.name}/{path.name}",
                "style": style,
                "style_folder": style_dir.name,
                "filename": path.name,
                "artist_slug": path.name.split("_", 1)[0],
                "join_key": normalize_key(f"{style_dir.name}/{path.name}"),
            })
    inventory = pd.DataFrame.from_records(records)
    return inventory.sort_values("relative_path").reset_index(drop=True)


DATASET_ROOT = validate_dataset_root(DATASET_ROOT)
inventory = build_inventory(DATASET_ROOT)
print(f"Dataset root : {DATASET_ROOT}")
print(f"Images on disk: {len(inventory):,}")
print(f"Style folders : {inventory['style'].nunique()}")
inventory.head()

## 2. Load metadata and reconcile against the inventory

### `classes.csv` — primary metadata
Columns, as verified by inspection:

| column | meaning |
|---|---|
| `filename` | `Style/artist_title.jpg` relative path |
| `artist` | artist name as text (lowercased); ~1,119 unique |
| `genre` | **actually the STYLE**, a list-literal, e.g. `['Impressionism']` |
| `description` | title slug (redundant with `filename`) |
| `phash` | perceptual hash (hex) — used for near-duplicate detection |
| `width`, `height` | pixel dimensions (from source metadata) |
| `genre_count` | number of style labels (1–3) |
| `subset` | `train` / `test` / `uncertain artist` |

### `wclasses.csv` — validation only
Columns `file, artist, genre, style`, where `artist`/`genre`/`style` are
**numeric IDs** in a shared integer namespace (artist 0–128, genre 129–139,
style 140–166). There is **no legend file**, so the artist and genre IDs are
unresolvable to names — reported, not guessed.

In [ ]:
def load_classes(project_root: Path) -> pd.DataFrame:
    """Load ``classes.csv``; parse the mislabelled ``genre`` column to styles."""
    df = pd.read_csv(project_root / "classes.csv")

    def parse_labels(raw: str) -> list[str]:
        try:
            value = ast.literal_eval(raw)
            return list(value) if isinstance(value, (list, tuple)) else [str(value)]
        except (ValueError, SyntaxError):
            return [raw]

    df["style_labels"] = df["genre"].apply(parse_labels)
    df["style_label"] = df["style_labels"].str[0]
    df["join_key"] = df["filename"].apply(normalize_key)
    return df


def load_wclasses(project_root: Path) -> pd.DataFrame:
    """Load ``wclasses.csv`` (numeric-ID file, used only for validation)."""
    df = pd.read_csv(project_root / "wclasses.csv")
    df["join_key"] = df["file"].apply(normalize_key)
    return df


classes = load_classes(DATASET_ROOT)
wclasses = load_wclasses(DATASET_ROOT)
print(f"classes.csv : {len(classes):,} rows, {classes['artist'].nunique()} artists")
print(f"wclasses.csv: {len(wclasses):,} rows, "
      f"{wclasses['style'].nunique()} style IDs, "
      f"{wclasses['artist'].nunique()} artist IDs, "
      f"{wclasses['genre'].nunique()} genre IDs")
classes[["filename", "artist", "style_label", "genre_count", "subset",
         "width", "height"]].head()

### Verify the style source-of-truth claim
Before trusting the folder as the style label, confirm it agrees with
`classes.csv` on every overlapping row.

In [ ]:
_check = classes.copy()
_check["folder_style"] = _check["filename"].str.split("/").str[0].str.replace("_", " ")
_mismatch = _check[_check["folder_style"] != _check["style_label"]]
print(f"Rows where folder style != classes.csv style label: {len(_mismatch)}")
assert len(_mismatch) == 0, "Folder is NOT a reliable style source — investigate!"
print("Confirmed: folder name is a reliable style label (0 mismatches).")

In [ ]:
CLASS_META = ["artist", "style_label", "style_labels", "phash",
              "width", "height", "genre_count", "subset"]


@dataclass
class Reconciliation:
    """Result of reconciling the filesystem against the two metadata files."""
    inventory: pd.DataFrame
    exact_matches: int
    fallback_matches: int
    inventory_without_classes: pd.DataFrame
    classes_without_inventory: pd.DataFrame
    inventory_without_wclasses: pd.DataFrame
    residual_key_collisions: pd.DataFrame
    notes: dict[str, object] = field(default_factory=dict)

    @property
    def matched(self) -> int:
        """Total disk images enriched with classes.csv metadata."""
        return self.exact_matches + self.fallback_matches


def reconcile(inventory: pd.DataFrame, classes: pd.DataFrame,
              wclasses: pd.DataFrame) -> Reconciliation:
    """Enrich the authoritative inventory with classes.csv metadata.

    A two-stage join is used, which is both more correct and easier to reason
    about than a single fuzzy join:

    1. **Exact** join on the full relative path -- unambiguous, covers the bulk.
    2. **Fallback** join on the encoding-robust ``join_key``, applied *only* to
       images still unmatched after stage 1 (the accented filenames). Limiting
       the fuzzy match to the residual avoids ever reassigning metadata for an
       image that already matched exactly, and sidesteps the within-folder key
       collisions (e.g. ``(1)`` vs ``-1`` filename variants) that a global fuzzy
       join would introduce.

    The inventory is never shrunk; unmatched rows on either side are reported.
    """
    exact_src = classes.drop_duplicates("filename").set_index("filename")[CLASS_META]
    merged = inventory.join(exact_src, on="relative_path")
    exact_mask = merged["artist"].notna()

    matched_paths = set(inventory["relative_path"]) & set(classes["filename"])
    classes_res = classes.loc[~classes["filename"].isin(matched_paths)]
    residual_collisions = classes_res[classes_res["join_key"].duplicated(keep=False)]
    fallback_src = classes_res.drop_duplicates("join_key").set_index("join_key")[CLASS_META]

    residual = merged.loc[~exact_mask].join(fallback_src, on="join_key", rsuffix="_fb")
    fallback_mask = residual["artist_fb"].notna()
    for col in CLASS_META:
        merged.loc[residual.index[fallback_mask], col] = \
            residual.loc[fallback_mask, f"{col}_fb"].values

    has_class = merged["artist"].notna()
    in_w = merged["join_key"].isin(set(wclasses["join_key"]))
    inv_keys = set(inventory["join_key"])

    return Reconciliation(
        inventory=merged,
        exact_matches=int(exact_mask.sum()),
        fallback_matches=int(fallback_mask.sum()),
        inventory_without_classes=merged.loc[~has_class, ["relative_path", "style"]],
        classes_without_inventory=classes.loc[
            ~classes["filename"].isin(matched_paths)
            & ~classes["join_key"].isin(inv_keys), ["filename"]],
        inventory_without_wclasses=merged.loc[~in_w, ["relative_path", "style"]],
        residual_key_collisions=residual_collisions[["filename", "join_key"]],
        notes={
            "disk_images": int(len(inventory)),
            "classes_rows": int(len(classes)),
            "wclasses_rows": int(len(wclasses)),
            "exact_matches": int(exact_mask.sum()),
            "fallback_matches": int(fallback_mask.sum()),
            "matched_to_classes": int(has_class.sum()),
            "matched_to_wclasses": int(in_w.sum()),
        },
    )


recon = reconcile(inventory, classes, wclasses)
inv = recon.inventory  # inventory enriched with classes.csv metadata
for k, v in recon.notes.items():
    print(f"{k:>20}: {v:,}")

**Reconciliation read-out.** Stage 1 (`exact_matches`) matches on the full path;
stage 2 (`fallback_matches`) rescues the accented filenames that the two
mojibake encodings would otherwise hide. Their sum equals `matched_to_classes`,
which -- because the fallback is confined to the residual -- lands at exactly
the `classes.csv` row count with **zero** residual key collisions (verified
below). Every disk image not matched exists only in `wclasses.csv`; it keeps its
filesystem style label and is reported, never dropped.

In [ ]:
# Quantify the encoding rescue and confirm the join is unambiguous.
print(f"Stage 1 exact-path matches      : {recon.exact_matches:,}")
print(f"Stage 2 fallback (accented)     : {recon.fallback_matches:,}")
print(f"Total enriched                  : {recon.matched:,}"
      f"  (classes.csv rows: {len(classes):,})")
print(f"Residual key collisions         : {recon.residual_key_collisions['join_key'].nunique()}")
print(f"classes.csv rows with no image  : {len(recon.classes_without_inventory)}")
print(f"Disk images w/o classes.csv     : {len(recon.inventory_without_classes):,}"
      "  (present only in wclasses.csv)")
print(f"Disk images missing from wclasses: {len(recon.inventory_without_wclasses)}")
print("\nExample on-disk images with no classes.csv metadata:")
display(recon.inventory_without_classes.head())

## 3. Image-integrity scan (graceful corruption handling)

Every image is checked with a two-stage test: `verify()` catches structural
corruption cheaply, then a full `load()` catches truncated pixel data that
`verify()` misses. Failures are recorded (path + error) and the scan continues
— it never raises. Set `INTEGRITY_SAMPLE` in Section 0 for a fast partial pass.

In [ ]:
def _check_one(project_root: Path, rel: str) -> dict[str, object]:
    """Two-stage integrity check for a single image; never raises."""
    abs_path = project_root / rel
    ok, error, mode, fmt = True, "", "", ""
    try:
        with Image.open(abs_path) as img:
            img.verify()
        with Image.open(abs_path) as img:  # re-open: verify() exhausts the file
            img.load()
            mode, fmt = img.mode, img.format or ""
    except Exception as exc:  # report every failure kind, never crash
        ok, error = False, f"{type(exc).__name__}: {exc}"
    return {"relative_path": rel, "ok": ok, "error": error, "mode": mode,
            "format": fmt}


def scan_image_integrity(inventory: pd.DataFrame, project_root: Path,
                         sample: int | None = None,
                         workers: int = 16) -> pd.DataFrame:
    """Verify each image opens and decodes; never raise on a bad file.

    ``verify()`` catches structural corruption cheaply; a follow-up ``load()``
    catches truncated pixel data it misses. The scan is I/O-bound, so images are
    checked across a thread pool for throughput on large datasets. Returns one
    row per image with an ``ok`` flag, error text, and decoded colour ``mode``.
    """
    paths = inventory["relative_path"]
    if sample is not None and sample < len(paths):
        idx = RNG.choice(len(paths), size=sample, replace=False)
        paths = paths.iloc[np.sort(idx)]
    paths = list(paths)

    with ThreadPoolExecutor(max_workers=workers) as pool:
        records = list(tqdm(
            pool.map(lambda rel: _check_one(project_root, rel), paths),
            total=len(paths), desc="Verifying images"))
    return pd.DataFrame.from_records(records)


integrity = scan_image_integrity(inv, DATASET_ROOT, sample=INTEGRITY_SAMPLE)
n_bad = int((~integrity["ok"]).sum())
print(f"Checked : {len(integrity):,} images")
print(f"Corrupt : {n_bad:,}")
print(f"Colour modes: {integrity['mode'].value_counts().to_dict()}")
if n_bad:
    display(integrity.loc[~integrity["ok"], ["relative_path", "error"]])

## 4. Duplicate detection (required: duplicate filenames)

Two independent notions of "duplicate":
- **Duplicate basename** — the same file name appearing under more than one
  style folder (the case the assignment asks to detect). The full path stays
  unique; the basename does not.
- Perceptual (visual) duplicates are handled separately as an optional
  enhancement in Section 7.

In [ ]:
def find_duplicate_filenames(inventory: pd.DataFrame) -> pd.DataFrame:
    """Return images whose basename occurs under more than one style folder."""
    dup_mask = inventory["filename"].duplicated(keep=False)
    return (inventory.loc[dup_mask, ["filename", "style", "relative_path"]]
            .sort_values(["filename", "style"]).reset_index(drop=True))


dup_names = find_duplicate_filenames(inv)
print(f"Duplicate basenames: {dup_names['filename'].nunique()} names "
      f"across {len(dup_names)} rows")
dup_names.head(10)

## 5. Summary statistics

In [ ]:
def summarize(inventory: pd.DataFrame, integrity: pd.DataFrame) -> pd.Series:
    """Compute the headline dataset statistics."""
    hd = inventory.dropna(subset=["width", "height"]).copy()
    hd["megapixels"] = hd["width"] * hd["height"] / 1e6
    hd["aspect"] = hd["width"] / hd["height"]
    return pd.Series({
        "images": len(inventory),
        "styles": inventory["style"].nunique(),
        "artists_named": inventory["artist"].dropna().nunique(),
        "corrupt_images": int((~integrity["ok"]).sum()),
        "with_classes_metadata": int(inventory["artist"].notna().sum()),
        "median_megapixels": round(float(hd["megapixels"].median()), 2),
        "median_aspect_ratio": round(float(hd["aspect"].median()), 3),
        "multi_label_images": int((inventory["genre_count"] > 1).sum()),
    })


summary = summarize(inv, integrity)
summary

In [ ]:
print("Images per style:")
display(inv["style"].value_counts().to_frame("images"))
print("Official subset split:")
display(inv["subset"].value_counts(dropna=False).to_frame("images"))

### Publication-quality visualizations

In [ ]:
# 1. Images per style
counts = inv["style"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(counts.index, counts.values, color="#3b6ea5")
ax.set_xlabel("Number of images")
ax.set_title("Images per style (filesystem inventory)")
for y, v in enumerate(counts.values):
    ax.text(v + 60, y, f"{v:,}", va="center", fontsize=8)
plt.show()

In [ ]:
# 2. Top 25 artists by image count (named via classes.csv)
top = inv["artist"].dropna().value_counts().head(25).sort_values()
fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(top.index, top.values, color="#8c564b")
ax.set_xlabel("Number of images")
ax.set_title("Top 25 artists by image count")
plt.show()

In [ ]:
# 3. Official train / test / uncertain split
sub = inv["subset"].fillna("(no metadata)").value_counts()
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(sub.index.astype(str), sub.values,
       color=["#2ca02c", "#ff7f0e", "#9467bd", "#7f7f7f"][:len(sub)])
ax.set_ylabel("Number of images")
ax.set_title("Official split (subset column)")
for x, v in enumerate(sub.values):
    ax.text(x, v + 200, f"{v:,}", ha="center", fontsize=9)
plt.show()

In [ ]:
# 4. Resolution and aspect-ratio distributions
hd = inv.dropna(subset=["width", "height"]).copy()
hd["mp"] = hd["width"] * hd["height"] / 1e6
hd["aspect"] = hd["width"] / hd["height"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(hd["mp"].clip(upper=15), bins=60, color="#3b6ea5")
axes[0].set(xlabel="Megapixels (clipped at 15)", ylabel="Images",
            title="Resolution distribution")
axes[1].hist(hd["aspect"].clip(0.2, 4), bins=60, color="#d62728")
axes[1].axvline(1.0, color="k", ls="--", lw=1)
axes[1].set(xlabel="Aspect ratio w/h (clipped)", title="Aspect-ratio distribution")
plt.show()

In [ ]:
# 5. Multi-label style frequency
ml = inv["genre_count"].dropna().astype(int).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(ml.index.astype(str), ml.values, color="#17becf")
ax.set_yscale("log")
ax.set(xlabel="Number of style labels", ylabel="Images (log scale)",
       title="Multi-label style frequency")
for x, v in enumerate(ml.values):
    ax.text(x, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
plt.show()

## 6. Reproducible visual spot-check

A fixed-seed random sample of images is rendered so a reviewer can eyeball the
data. The same seed (Section 0) reproduces the identical sample every run.

In [ ]:
def show_sample(inventory: pd.DataFrame, project_root: Path, n: int = 12,
                cols: int = 4) -> None:
    """Render a reproducible random image sample with style captions."""
    picks = inventory.sample(n=n, random_state=RANDOM_SEED)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    for ax, (_, r) in zip(axes.ravel(), picks.iterrows()):
        try:
            with Image.open(project_root / r["relative_path"]) as im:
                ax.imshow(im.convert("RGB"))
        except Exception as exc:
            ax.text(0.5, 0.5, f"unreadable\n{exc}", ha="center", va="center")
        ax.set_title(r["style"], fontsize=9)
        ax.axis("off")
    for ax in axes.ravel()[n:]:
        ax.axis("off")
    fig.suptitle(f"Reproducible sample (seed={RANDOM_SEED})", y=1.01)
    plt.show()


show_sample(inv, DATASET_ROOT)

---
# 7. Optional enhancements

Everything required by the assessment is complete above. The following is an
**optional** analysis, kept separate and clearly labelled.

## 7.1 Perceptual-hash near-duplicate detection

Duplicate *basenames* (Section 4) only catch identical filenames. Genuinely
duplicated **artwork** can appear under different names or styles. The `phash`
column already in `classes.csv` lets us group visually identical images with
no additional dependency. Images without `classes.csv` metadata carry no hash
and are excluded from this optional check.

In [ ]:
def find_phash_duplicates(inventory: pd.DataFrame) -> pd.DataFrame:
    """Group images that share an identical perceptual hash."""
    hashed = inventory.dropna(subset=["phash"])
    counts = hashed.groupby("phash")["relative_path"].transform("size")
    groups = hashed.loc[counts > 1, ["phash", "style", "relative_path"]]
    return groups.sort_values(["phash", "relative_path"]).reset_index(drop=True)


phash_dups = find_phash_duplicates(inv)
print(f"Perceptual-hash duplicate groups: {phash_dups['phash'].nunique()} "
      f"covering {len(phash_dups)} images")
cross = (phash_dups.groupby('phash')['style'].nunique() > 1).sum()
print(f"Of those, {cross} groups span more than one style folder.")
phash_dups.head(10)

## Assumptions & limitations (explicitly stated)

- **No ID legend exists** for `wclasses.csv`; artist and genre integer IDs are
  therefore left unresolved. The 27-way `style` IDs *could* be mapped to names
  via folder co-occurrence, but that mapping is not required for any deliverable
  and is not asserted as fact here.
- The **encoding-robust fallback key** could in principle create false matches
  by collapsing distinct paths. This risk is *verified* away, not assumed:
  because the fallback is confined to images unmatched exactly, its residual key
  collisions are **0** (printed in Section 2). Cross-folder duplicate basenames
  do not collide because the key includes the style folder.
- `width`/`height` are taken from `classes.csv` metadata rather than re-measured
  from pixels, for efficiency. Images lacking `classes.csv` metadata are absent
  from resolution statistics (reported as such).
- `phash` near-duplicate detection covers only images present in `classes.csv`;
  the 22 groups it finds match the duplicate perceptual hashes in `classes.csv`
  itself, independent of any join.
- The dataset is a known WikiArt export; no external labels are assumed beyond
  the two provided CSVs and the folder structure.